In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [98]:
PJME_data = pd.read_csv("processed_data/PJME_phase2_preprocessed.csv",parse_dates=["Datetime"])

In [99]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [5]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print("\nTrain period:")
print(train.index.min(), "to", train.index.max())

print("\nValidation period:")
print(validation.index.min(), "to", validation.index.max())

print("\nTest period:")
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)

Train period:
2002-01-08 01:00:00 to 2013-08-13 16:00:00

Validation period:
2013-08-13 17:00:00 to 2016-02-07 08:00:00

Test period:
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [6]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [7]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = target_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = target_scaler.transform(
    y_test.values.reshape(-1, 1)
)

print("Scaling completed.")

Scaling completed.


In [8]:
def create_sequences(X, y, input_steps=168, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(
            X[i-input_steps:i]
        )

        y_seq.append(
            y[i:i+output_steps].flatten()
        )

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=168,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=168,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=168,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101465, 168, 15)
y_train: (101465, 24)
X_val: (21593, 168, 15)
y_val: (21593, 24)
X_test: (21593, 168, 15)
y_test: (21593, 24)


**Early Stopping**

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [10]:
rnn_es_model = Sequential([
    SimpleRNN(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),
    Dense(24)
])

rnn_es_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_es_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [12]:
history_es = rnn_es_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1281 - mae: 0.2559 - val_loss: 0.0942 - val_mae: 0.2261
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0788 - mae: 0.2027 - val_loss: 0.0878 - val_mae: 0.2154
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0721 - mae: 0.1923 - val_loss: 0.0842 - val_mae: 0.2094
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0688 - mae: 0.1872 - val_loss: 0.0787 - val_mae: 0.2004
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0663 - mae: 0.1832 - val_loss: 0.0786 - val_mae: 0.2024
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0648 - mae: 0.1807 - val_loss: 0.0814 - val_mae: 0.2013
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0639 - mae: 0.1792 - val_loss: 0.0795 - val_mae: 0.2030
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0630 - mae: 0.1778 - val_loss: 0.0807 - val_mae: 0.2058
Epoch 8: early stopping


In [13]:
rnn_es_pred_scaled = rnn_es_model.predict(
    X_test_seq,
    batch_size=64
)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [14]:
rnn_es_pred = target_scaler.inverse_transform(
    rnn_es_pred_scaled.reshape(-1, 1)
).reshape(rnn_es_pred_scaled.shape)

rnn_es_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(
    rnn_es_actual.flatten(),
    rnn_es_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        rnn_es_actual.flatten(),
        rnn_es_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (rnn_es_actual.flatten() - rnn_es_pred.flatten())
        / rnn_es_actual.flatten()
    )
) * 100

r2 = r2_score(
    rnn_es_actual.flatten(),
    rnn_es_pred.flatten()
)

bias = np.mean(
    rnn_es_pred.flatten() - rnn_es_actual.flatten()
)

print("==== RNN + EARLY STOPPING ====")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

==== RNN + EARLY STOPPING ====
MAE : 1442.4374170896492
RMSE: 1988.3840351875392
MAPE: 4.57080183304137
R²  : 0.9046868337417405
Bias: 85.23315503556442


In [16]:
print("Epochs completed:", len(history_es.history["loss"]))

Epochs completed: 8


**Dropout**

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

rnn_dropout_model = Sequential([
    SimpleRNN(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(64, activation="relu"),

    Dropout(0.2),

    Dense(24)
])

rnn_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_dropout_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
history_dropout = rnn_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2022 - mae: 0.3365 - val_loss: 0.0996 - val_mae: 0.2323
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.1324 - mae: 0.2756 - val_loss: 0.0945 - val_mae: 0.2278
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1254 - mae: 0.2676 - val_loss: 0.0926 - val_mae: 0.2231
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1212 - mae: 0.2628 - val_loss: 0.0901 - val_mae: 0.2193
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1191 - mae: 0.2602 - val_loss: 0.0879 - val_mae: 0.2153
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1175 - mae: 0.2581 - val_loss: 0.0896 - val_mae: 0.2184
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1158 - mae: 0.2563 - val_loss: 0.0871 - val_mae: 0.2147
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1145 - mae: 0.2547 - val_loss: 0.0927 - val_mae: 0.2213
Epoch 9/15
1586/1586 ━━━

In [19]:
dropout_pred_scaled = rnn_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [20]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        dropout_actual.flatten(),
        dropout_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten()
    )
) * 100

r2 = r2_score(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

bias = np.mean(
    dropout_pred.flatten() -
    dropout_actual.flatten()
)

print("=== RNN + DROPOUT ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== RNN + DROPOUT ===
MAE : 1456.016174924129
RMSE: 2007.4260748113516
MAPE: 4.65354036516522
R²  : 0.9028525324676241
Bias: 206.21227118115053


**Batch Normalization**

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, BatchNormalization

rnn_bn_model = Sequential([
    SimpleRNN(
        64,
        input_shape=(168, 15)
    ),

    BatchNormalization(),

    Dense(64, activation="relu"),

    BatchNormalization(),

    Dense(24)
])

rnn_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_bn_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,352 (44.34 KB)

 Trainable params: 11,096 (43.34 KB)

 Non-trainable params: 256 (1.00 KB)

In [23]:
history_bn = rnn_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1714 - mae: 0.3031 - val_loss: 0.1002 - val_mae: 0.2346
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1007 - mae: 0.2377 - val_loss: 0.0938 - val_mae: 0.2244
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0928 - mae: 0.2271 - val_loss: 0.0835 - val_mae: 0.2128
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0882 - mae: 0.2209 - val_loss: 0.0837 - val_mae: 0.2080
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0860 - mae: 0.2178 - val_loss: 0.0867 - val_mae: 0.2109
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0837 - mae: 0.2146 - val_loss: 0.0797 - val_mae: 0.2011
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0831 - mae: 0.2138 - val_loss: 0.0783 - val_mae: 0.1980
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0817 - mae: 0.2118 - val_loss: 0.0767 - val_mae: 0.1984
Epoch 9/15
1586/1586 ━━━

In [24]:
bn_pred_scaled = rnn_bn_model.predict(
    X_test_seq,
    batch_size=64
)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [25]:
rnn_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(rnn_es_pred_scaled.shape)

rnn_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(
    rnn_bn_actual.flatten(),
    rnn_bn_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        rnn_bn_actual.flatten(),
        rnn_bn_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (rnn_bn_actual.flatten() - rnn_bn_pred.flatten())
        / rnn_bn_actual.flatten()
    )
) * 100

r2 = r2_score(
    rnn_bn_actual.flatten(),
    rnn_bn_pred.flatten()
)

bias = np.mean(
    rnn_bn_pred.flatten() - rnn_bn_actual.flatten()
)

print("==== RNN + Batch ====")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

==== RNN + Batch ====
MAE : 1449.4259604726383
RMSE: 2022.7581740055055
MAPE: 4.600892048195033
R²  : 0.9013629008438131
Bias: 217.97801266041319


**ADAM**

In [27]:
from tensorflow.keras.optimizers import Adam

rnn_adam = Sequential([
    SimpleRNN(64, input_shape=(168, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_adam.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [28]:
history_adam = rnn_adam.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1269 - mae: 0.2549 - val_loss: 0.0917 - val_mae: 0.2192
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0779 - mae: 0.2008 - val_loss: 0.0881 - val_mae: 0.2157
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0725 - mae: 0.1922 - val_loss: 0.0828 - val_mae: 0.2049
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0692 - mae: 0.1867 - val_loss: 0.0864 - val_mae: 0.2126
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0675 - mae: 0.1838 - val_loss: 0.0775 - val_mae: 0.1986
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0657 - mae: 0.1808 - val_loss: 0.0796 - val_mae: 0.2016
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0646 - mae: 0.1792 - val_loss: 0.0784 - val_mae: 0.2002
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0635 - mae: 0.1776 - val_loss: 0.0786 - val_mae: 0.2003
Epoch 9/15
1586/1586 ━━━

In [29]:
adam_pred_scaled = rnn_adam.predict(
    X_test_seq,
    batch_size=64
)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [30]:
adam_pred = target_scaler.inverse_transform(
    adam_pred_scaled.reshape(-1, 1)
).reshape(adam_pred_scaled.shape)

adam_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [31]:
adam_mae = mean_absolute_error(
    adam_actual.flatten(),
    adam_pred.flatten()
)

adam_rmse = np.sqrt(
    mean_squared_error(
        adam_actual.flatten(),
        adam_pred.flatten()
    )
)

adam_mape = np.mean(
    np.abs(
        (adam_actual.flatten() - adam_pred.flatten())
        / adam_actual.flatten()
    )
) * 100

adam_r2 = r2_score(
    adam_actual.flatten(),
    adam_pred.flatten()
)

adam_bias = np.mean(
    adam_pred.flatten() -
    adam_actual.flatten()
)

print("========== RNN + ADAM ==========")
print("MAE :", adam_mae)
print("RMSE:", adam_rmse)
print("MAPE:", adam_mape)
print("R²  :", adam_r2)
print("Bias:", adam_bias)

========== RNN + ADAM ==========
MAE : 1401.9271355353092
RMSE: 1956.949688716373
MAPE: 4.466173715600929
R²  : 0.9076766227592874
Bias: 275.53451501104354


**RMSPROP**

In [32]:
from tensorflow.keras.optimizers import RMSprop

rnn_rmsprop = Sequential([
    SimpleRNN(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_rmsprop.compile(
    optimizer=RMSprop(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

rnn_rmsprop.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
history_rmsprop = rnn_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1258 - mae: 0.2565 - val_loss: 0.0957 - val_mae: 0.2264
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0804 - mae: 0.2042 - val_loss: 0.0888 - val_mae: 0.2143
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0739 - mae: 0.1937 - val_loss: 0.0845 - val_mae: 0.2093
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0706 - mae: 0.1881 - val_loss: 0.0880 - val_mae: 0.2117
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0683 - mae: 0.1843 - val_loss: 0.0790 - val_mae: 0.2020
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0666 - mae: 0.1817 - val_loss: 0.0839 - val_mae: 0.2076
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0653 - mae: 0.1793 - val_loss: 0.0773 - val_mae: 0.1995
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0644 - mae: 0.1779 - val_loss: 0.0827 - val_mae: 0.2065
Epoch 9/15
1586/1586 ━━━

In [34]:
rmsprop_pred_scaled = rnn_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21593, 24)


In [35]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [36]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rmsprop_mae = mean_absolute_error(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_rmse = np.sqrt(
    mean_squared_error(
        rmsprop_actual.flatten(),
        rmsprop_pred.flatten()
    )
)

rmsprop_mape = np.mean(
    np.abs(
        (rmsprop_actual.flatten() - rmsprop_pred.flatten())
        / rmsprop_actual.flatten()
    )
) * 100

rmsprop_r2 = r2_score(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_bias = np.mean(
    rmsprop_pred.flatten() -
    rmsprop_actual.flatten()
)

print("==== RNN + RMSPROP ====")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

==== RNN + RMSPROP ====
MAE : 1431.3565403170057
RMSE: 1999.2467893785217
MAPE: 4.543059594197409
R²  : 0.9036425770705
Bias: 193.86489551253106


**SGD**

In [41]:
from tensorflow.keras.optimizers import SGD

rnn_sgd = Sequential([
    SimpleRNN(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_sgd.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="mse",
    metrics=["mae"]
)

rnn_sgd.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_7 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
history_sgd = rnn_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.4296 - mae: 0.4994 - val_loss: 0.2155 - val_mae: 0.3646
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.1733 - mae: 0.3205 - val_loss: 0.1615 - val_mae: 0.3086
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.1388 - mae: 0.2833 - val_loss: 0.1422 - val_mae: 0.2869
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.1246 - mae: 0.2664 - val_loss: 0.1312 - val_mae: 0.2738
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1162 - mae: 0.2557 - val_loss: 0.1243 - val_mae: 0.2649
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1102 - mae: 0.2480 - val_loss: 0.1189 - val_mae: 0.2582
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.1056 - mae: 0.2419 - val_loss: 0.1151 - val_mae: 0.2530
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.1016 - mae: 0.2364 - val_loss: 0.1109 - val_mae: 0.2476
Epoch 9/15
1586/1586 ━━━

In [43]:
sgd_pred_scaled = rnn_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Prediction shape: (21593, 24)


In [44]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sgd_mae = mean_absolute_error(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_rmse = np.sqrt(
    mean_squared_error(
        sgd_actual.flatten(),
        sgd_pred.flatten()
    )
)

sgd_mape = np.mean(
    np.abs(
        (sgd_actual.flatten() - sgd_pred.flatten())
        / sgd_actual.flatten()
    )
) * 100

sgd_r2 = r2_score(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_bias = np.mean(
    sgd_pred.flatten() -
    sgd_actual.flatten()
)
print("=== RNN + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== RNN + SGD ===
MAE : 1587.0868063796715
RMSE: 2166.9122430922744
MAPE: 5.075387416706246
R²  : 0.8868029761759146
Bias: 227.20396023915205


**Learning Rate Scheduling**

In [46]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

rnn_lr = Sequential([
    SimpleRNN(
        64,
        input_shape=(168, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [47]:
history_lr = rnn_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.1239 - mae: 0.2552 - val_loss: 0.0955 - val_mae: 0.2273 - learning_rate: 0.0010
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0808 - mae: 0.2056 - val_loss: 0.0906 - val_mae: 0.2167 - learning_rate: 0.0010
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0747 - mae: 0.1954 - val_loss: 0.0911 - val_mae: 0.2199 - learning_rate: 0.0010
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0713 - mae: 0.1898 - val_loss: 0.0846 - val_mae: 0.2091 - learning_rate: 0.0010
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0687 - mae: 0.1854 - val_loss: 0.0849 - val_mae: 0.2101 - learning_rate: 0.0010
Epoch 6/15
1582/1586 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0669 - mae: 0.1825
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0669 - mae: 0.1825 - val_loss: 0.0877 - val_mae: 0.214

In [48]:
lr_pred_scaled = rnn_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21593, 24)


In [49]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr_mae = mean_absolute_error(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_rmse = np.sqrt(
    mean_squared_error(
        lr_actual.flatten(),
        lr_pred.flatten()
    )
)

lr_mape = np.mean(
    np.abs(
        (lr_actual.flatten() - lr_pred.flatten())
        / lr_actual.flatten()
    )
) * 100

lr_r2 = r2_score(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_bias = np.mean(
    lr_pred.flatten() -
    lr_actual.flatten()
)

print("==== RNN + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== RNN + RMSPROP + LearningRate ====
MAE : 1391.4736401993944
RMSE: 1955.1376315342836
MAPE: 4.431959790886786
R²  : 0.9078475191143814
Bias: 260.42541081986445


**layers, neurons**

In [51]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_deep = Sequential([

    SimpleRNN(
        64,
        return_sequences=True,
        input_shape=(168, 15)
    ),

    SimpleRNN(
        32,
        return_sequences=False
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

rnn_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_deep.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_9 (SimpleRNN)        │ (None, 168, 64)        │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_10 (SimpleRNN)       │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,896 (46.47 KB)

 Trainable params: 11,896 (46.47 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
history_deep = rnn_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 43s 27ms/step - loss: 0.1301 - mae: 0.2585 - val_loss: 0.0924 - val_mae: 0.2240
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0790 - mae: 0.2036 - val_loss: 0.0853 - val_mae: 0.2126
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0722 - mae: 0.1925 - val_loss: 0.0825 - val_mae: 0.2071
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - loss: 0.0691 - mae: 0.1875 - val_loss: 0.0801 - val_mae: 0.2042
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0670 - mae: 0.1840 - val_loss: 0.0792 - val_mae: 0.2012
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0651 - mae: 0.1811 - val_loss: 0.0807 - val_mae: 0.2088
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0640 - mae: 0.1795 - val_loss: 0.0817 - val_mae: 0.2078
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0633 - mae: 0.1782 - val_loss: 0.0798 - val_mae: 0.2029
Epoch 9/15
1586/1586 ━━━

In [53]:
deep_pred_scaled = rnn_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step
Prediction shape: (21593, 24)


In [54]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [55]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

deep_mae = mean_absolute_error(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_rmse = np.sqrt(
    mean_squared_error(
        deep_actual.flatten(),
        deep_pred.flatten()
    )
)

deep_mape = np.mean(
    np.abs(
        (deep_actual.flatten() - deep_pred.flatten())
        / deep_actual.flatten()
    )
) * 100

deep_r2 = r2_score(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_bias = np.mean(
    deep_pred.flatten() -
    deep_actual.flatten()
)

print("========== DEEP RNN ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

========== DEEP RNN ==========
MAE : 1463.0599539297493
RMSE: 2080.42019144102
MAPE: 4.623851630921524
R²  : 0.8956591231413713
Bias: 287.09569205596506


In [56]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_128 = Sequential([

    SimpleRNN(
        128,
        input_shape=(168, 15)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

rnn_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_128.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_11 (SimpleRNN)       │ (None, 128)            │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,248 (110.34 KB)

 Trainable params: 28,248 (110.34 KB)

 Non-trainable params: 0 (0.00 B)

In [57]:
history_128 = rnn_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 43s 27ms/step - loss: 0.1083 - mae: 0.2357 - val_loss: 0.0880 - val_mae: 0.2165
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0730 - mae: 0.1932 - val_loss: 0.0902 - val_mae: 0.2192
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0679 - mae: 0.1848 - val_loss: 0.0855 - val_mae: 0.2084
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0654 - mae: 0.1808 - val_loss: 0.0757 - val_mae: 0.1956
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0632 - mae: 0.1772 - val_loss: 0.0794 - val_mae: 0.1999
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0621 - mae: 0.1757 - val_loss: 0.0765 - val_mae: 0.1935
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0608 - mae: 0.1737 - val_loss: 0.0742 - val_mae: 0.1955
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - loss: 0.0587 - mae: 0.1710 - val_loss: 0.0824 - val_mae: 0.2045
Epoch 9/15
1586/1586 ━━━

In [58]:
rnn128_pred_scaled = rnn_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn128_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step
Prediction shape: (21593, 24)


In [59]:
rnn128_pred = target_scaler.inverse_transform(
    rnn128_pred_scaled.reshape(-1, 1)
).reshape(rnn128_pred_scaled.shape)

rnn128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [60]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnn128_mae = mean_absolute_error(
    rnn128_actual.flatten(),
    rnn128_pred.flatten()
)

rnn128_rmse = np.sqrt(
    mean_squared_error(
        rnn128_actual.flatten(),
        rnn128_pred.flatten()
    )
)

rnn128_mape = np.mean(
    np.abs(
        (rnn128_actual.flatten() - rnn128_pred.flatten())
        / rnn128_actual.flatten()
    )
) * 100

rnn128_r2 = r2_score(
    rnn128_actual.flatten(),
    rnn128_pred.flatten()
)

rnn128_bias = np.mean(
    rnn128_pred.flatten() -
    rnn128_actual.flatten()
)

print("=== RNN 128 NEURONS ===")
print("MAE :", rnn128_mae)
print("RMSE:", rnn128_rmse)
print("MAPE:", rnn128_mape)
print("R²  :", rnn128_r2)
print("Bias:", rnn128_bias)

=== RNN 128 NEURONS ===
MAE : 1421.2446590304571
RMSE: 2045.7797521954965
MAPE: 4.48897927494526
R²  : 0.8991048910414456
Bias: 192.43200104890883


**32 neurons**

In [61]:
rnn_32 = Sequential([
    SimpleRNN(
        32,
        input_shape=(168, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_32.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_12 (SimpleRNN)       │ (None, 32)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,208 (20.34 KB)

 Trainable params: 5,208 (20.34 KB)

 Non-trainable params: 0 (0.00 B)

In [62]:
history_32 = rnn_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1509 - mae: 0.2812 - val_loss: 0.1047 - val_mae: 0.2372
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0885 - mae: 0.2177 - val_loss: 0.0948 - val_mae: 0.2215
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0798 - mae: 0.2044 - val_loss: 0.0922 - val_mae: 0.2191
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0753 - mae: 0.1972 - val_loss: 0.0853 - val_mae: 0.2103
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0728 - mae: 0.1930 - val_loss: 0.0845 - val_mae: 0.2061
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0713 - mae: 0.1902 - val_loss: 0.0854 - val_mae: 0.2078
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0698 - mae: 0.1876 - val_loss: 0.0840 - val_mae: 0.2055
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0689 - mae: 0.1862 - val_loss: 0.0801 - val_mae: 0.2008
Epoch 9/15
1586/1586 ━━━━━━━━━━━

In [63]:
rnn32_pred_scaled = rnn_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn32_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21593, 24)


In [64]:
rnn32_pred = target_scaler.inverse_transform(
    rnn32_pred_scaled.reshape(-1, 1)
).reshape(rnn32_pred_scaled.shape)

rnn32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [65]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnn32_mae = mean_absolute_error(
    rnn32_actual.flatten(),
    rnn32_pred.flatten()
)

rnn32_rmse = np.sqrt(
    mean_squared_error(
        rnn32_actual.flatten(),
        rnn32_pred.flatten()
    )
)

rnn32_mape = np.mean(
    np.abs(
        (rnn32_actual.flatten() - rnn32_pred.flatten())
        / rnn32_actual.flatten()
    )
) * 100

rnn32_r2 = r2_score(
    rnn32_actual.flatten(),
    rnn32_pred.flatten()
)

rnn32_bias = np.mean(
    rnn32_pred.flatten() -
    rnn32_actual.flatten()
)

print("=== RNN 32 NEURONS ===")
print("MAE :", rnn32_mae)
print("RMSE:", rnn32_rmse)
print("MAPE:", rnn32_mape)
print("R²  :", rnn32_r2)
print("Bias:", rnn32_bias)

=== RNN 32 NEURONS ===
MAE : 1467.1060833207605
RMSE: 2045.6708454485063
MAPE: 4.654971390277702
R²  : 0.899115633024414
Bias: 258.54255947459944


**batch size(32)**

In [66]:
rnn_batch32 = Sequential([
    SimpleRNN(64, input_shape=(168, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = rnn_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1111 - mae: 0.2397 - val_loss: 0.0890 - val_mae: 0.2148
Epoch 2/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0748 - mae: 0.1957 - val_loss: 0.0869 - val_mae: 0.2163
Epoch 3/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0697 - mae: 0.1876 - val_loss: 0.0804 - val_mae: 0.2043
Epoch 4/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0667 - mae: 0.1825 - val_loss: 0.0771 - val_mae: 0.1963
Epoch 5/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0650 - mae: 0.1798 - val_loss: 0.0822 - val_mae: 0.2053
Epoch 6/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0639 - mae: 0.1780 - val_loss: 0.0795 - val_mae: 0.1999
Epoch 7/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0631 - mae: 0.1769 - val_loss: 0.0762 - val_mae: 0.1972
Epoch 8/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0620 - mae: 0.1750 - val_loss: 0.0766 - val_mae: 0.1943
Epoch 9/15
3171/3171 ━━━━━━━━━━━

In [67]:
rnn_batch32_pred_scaled = rnn_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", rnn_batch32_pred_scaled.shape)

675/675 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Prediction shape: (21593, 24)


In [68]:
rnnbatch32_pred = target_scaler.inverse_transform(
    rnn_batch32_pred_scaled.reshape(-1, 1)
).reshape(rnn_batch32_pred_scaled.shape)

rnnbatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [69]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnnb32_mae = mean_absolute_error(
    rnnbatch32_actual.flatten(),
    rnnbatch32_pred.flatten()
)

rnnb32_rmse = np.sqrt(
    mean_squared_error(
        rnnbatch32_actual.flatten(),
        rnnbatch32_pred.flatten()
    )
)

rnnb32_mape = np.mean(
    np.abs(
        (rnnbatch32_actual.flatten() - rnnbatch32_pred.flatten())
        / rnnbatch32_actual.flatten()
    )
) * 100

rnnb32_r2 = r2_score(
    rnnbatch32_actual.flatten(),
    rnnbatch32_pred.flatten()
)

rnnb32_bias = np.mean(
    rnnbatch32_pred.flatten() -
    rnnbatch32_actual.flatten()
)

print("=== RNN batch32 NEURONS ===")
print("MAE :", rnnb32_mae)
print("RMSE:", rnnb32_rmse)
print("MAPE:", rnnb32_mape)
print("R²  :", rnnb32_r2)
print("Bias:", rnnb32_bias)

=== RNN batch32 NEURONS ===
MAE : 1407.5672331902024
RMSE: 1977.5205726395573
MAPE: 4.477985259160447
R²  : 0.9057254686072457
Bias: 311.5092754141123


**batchsize128**

In [70]:
rnn_batch128 = Sequential([
    SimpleRNN(64, input_shape=(168, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = rnn_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - loss: 0.1544 - mae: 0.2829 - val_loss: 0.1052 - val_mae: 0.2385
Epoch 2/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - loss: 0.0855 - mae: 0.2129 - val_loss: 0.0891 - val_mae: 0.2167
Epoch 3/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - loss: 0.0760 - mae: 0.1980 - val_loss: 0.0853 - val_mae: 0.2081
Epoch 4/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - loss: 0.0713 - mae: 0.1903 - val_loss: 0.0829 - val_mae: 0.2077
Epoch 5/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - loss: 0.0689 - mae: 0.1862 - val_loss: 0.0836 - val_mae: 0.2080
Epoch 6/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - loss: 0.0670 - mae: 0.1833 - val_loss: 0.0800 - val_mae: 0.2033
Epoch 7/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - loss: 0.0658 - mae: 0.1814 - val_loss: 0.0770 - val_mae: 0.1971
Epoch 8/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - loss: 0.0647 - mae: 0.1796 - val_loss: 0.0821 - val_mae: 0.2066
Epoch 9/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 

In [71]:
rnn_batch128_pred_scaled = rnn_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", rnn_batch128_pred_scaled.shape)

169/169 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Prediction shape: (21593, 24)


In [72]:
rnnbatch128_pred = target_scaler.inverse_transform(
    rnn_batch128_pred_scaled.reshape(-1, 1)
).reshape(rnn_batch128_pred_scaled.shape)

rnnbatch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [73]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnnb128_mae = mean_absolute_error(rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten())
rnnb128_rmse = np.sqrt(mean_squared_error(rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten()))
rnnb128_mape = np.mean(np.abs((rnnbatch128_actual.flatten() - rnnbatch128_pred.flatten()) / rnnbatch128_actual.flatten())) * 100
rnnb128_r2 = r2_score(rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten())
rnnb128_bias = np.mean(rnnbatch128_pred.flatten() - rnnbatch128_actual.flatten())

print("=== RNN batch128  ===")
print("MAE :", rnnb128_mae)
print("RMSE:", rnnb128_rmse)
print("MAPE:", rnnb128_mape)
print("R²  :", rnnb128_r2)
print("Bias:", rnnb128_bias)

=== RNN batch128  ===
MAE : 1392.5691757587274
RMSE: 1949.3553913471533
MAPE: 4.4230654213894045
R²  : 0.9083917875401955
Bias: 237.11039008364617


**HyperParameter Tuning**

In [83]:
pip install keras-tuner


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/aximsoft/tfenv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [88]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [89]:
def build_rnn(hp):

    model = Sequential()

    model.add(SimpleRNN(
            units=hp.Choice("rnn_units", values=[32, 64, 128]),
            input_shape=(168, 15)
        )
    )

    model.add(Dense(units=hp.Choice("dense_units", values=[32, 64, 128]), activation="relu"))
    model.add(Dense(24))

    learning_rate = hp.Choice("learning_rate", values=[0.0001, 0.0005, 0.001])

    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse", metrics=["mae"])

    return model

In [91]:
tuner = kt.RandomSearch(
    build_rnn,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="rnn_268_to_24"
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [92]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 03m 06s]
val_loss: 0.08189494162797928

Best val_loss So Far: 0.07256140559911728
Total elapsed time: 00h 35m 17s


In [93]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:", best_hp.get("rnn_units"))
print("Dense Units:", best_hp.get("dense_units"))
print("Learning Rate:", best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 64
Dense Units: 128
Learning Rate: 0.001


In [94]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,536 (64.59 KB)

 Trainable params: 16,536 (64.59 KB)

 Non-trainable params: 0 (0.00 B)

In [95]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0566 - mae: 0.1669 - val_loss: 0.0779 - val_mae: 0.1989
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0562 - mae: 0.1664 - val_loss: 0.0742 - val_mae: 0.1915
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0559 - mae: 0.1662 - val_loss: 0.0743 - val_mae: 0.1931
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0552 - mae: 0.1651 - val_loss: 0.0779 - val_mae: 0.1956
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0548 - mae: 0.1647 - val_loss: 0.0806 - val_mae: 0.2003
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0542 - mae: 0.1640 - val_loss: 0.0759 - val_mae: 0.1976
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0537 - mae: 0.1635 - val_loss: 0.0756 - val_mae: 0.1957
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0530 - mae: 0.1628 - val_loss: 0.0752 - val_mae: 0.1945
Epoch 9/15
1586/1586 ━━━

In [96]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [97]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
import numpy as np

best_mae = mean_absolute_error(best_actual.flatten(),best_pred.flatten())
best_rmse = np.sqrt(mean_squared_error(best_actual.flatten(), best_pred.flatten()))
best_mape = np.mean(np.abs((best_actual.flatten() - best_pred.flatten())/ best_actual.flatten()))*100
best_r2 = r2_score(best_actual.flatten(),best_pred.flatten())
best_bias = np.mean(best_pred.flatten() - best_actual.flatten())

print("HYPERPARAMETER TUNED RNN")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED RNN
MAE : 1420.3538599296503
RMSE: 2029.101753915076
MAPE: 4.500508471335718
R²  : 0.9007432583397899
Bias: 219.11408073055358


**additional lags**

In [103]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [104]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144552, 21)


In [105]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [106]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101186, 21)
Validation: (21683, 21)
Test: (21683, 21)


In [107]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [108]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [110]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=168,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=168,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=168,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (100995, 168, 21)
y_train: (100995, 24)
X_val: (21492, 168, 21)
y_val: (21492, 24)
X_test: (21492, 168, 21)
y_test: (21492, 24)


In [111]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_lags = Sequential([
    SimpleRNN(
        64,
        input_shape=(168, 21)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

rnn_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_lags.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,224 (43.84 KB)

 Trainable params: 11,224 (43.84 KB)

 Non-trainable params: 0 (0.00 B)

In [112]:
history_lags = rnn_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1184 - mae: 0.2454 - val_loss: 0.0927 - val_mae: 0.2205
Epoch 2/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0757 - mae: 0.1975 - val_loss: 0.0907 - val_mae: 0.2132
Epoch 3/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0709 - mae: 0.1897 - val_loss: 0.0823 - val_mae: 0.2105
Epoch 4/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0674 - mae: 0.1842 - val_loss: 0.0796 - val_mae: 0.1997
Epoch 5/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0659 - mae: 0.1814 - val_loss: 0.0746 - val_mae: 0.1935
Epoch 6/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0639 - mae: 0.1781 - val_loss: 0.0766 - val_mae: 0.1992
Epoch 7/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0628 - mae: 0.1766 - val_loss: 0.0762 - val_mae: 0.1978
Epoch 8/15
1579/1579 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0621 - mae: 0.1752 - val_loss: 0.0736 - val_mae: 0.1914
Epoch 9/15
1579/1579 ━━━

In [113]:
pred_scaled = rnn_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21492, 24)


In [114]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [115]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(
    actual.flatten(),
    pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        actual.flatten(),
        pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (actual.flatten() - pred.flatten())
        / actual.flatten()
    )
) * 100

r2 = r2_score(
    actual.flatten(),
    pred.flatten()
)

bias = np.mean(
    pred.flatten() - actual.flatten()
)

print("========== RNN + ADDITIONAL LAGS ==========")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

========== RNN + ADDITIONAL LAGS ==========
MAE : 1405.2472066353298
RMSE: 1974.0998561917397
MAPE: 4.474581326853382
R²  : 0.9061420038504027
Bias: 338.74023258018246


In [1]:
import pandas as pd
import os

rnn_168_24_results = [
    ["RNN Baseline", 168, 24, 1430.2133220007822, 1981.7279488063964, 4.582684822640543, 0.9053238845459299, 375.384035532258],
    ["RNN + EarlyStopping", 168, 24, 1442.4374170896492, 1988.3840351875392, 4.57080183304137, 0.9046868337417405, 85.23315503556442],
    ["RNN + Dropout", 168, 24, 1456.016174924129, 2007.4260748113516, 4.65354036516522, 0.9028525324676241, 206.21227118115053],
    ["RNN + Batch Normalization", 168, 24, 1449.4259604726383, 2022.7581740055055, 4.600892048195033, 0.9013629008438131, 217.97801266041319],
    ["RNN + Adam", 168, 24, 1401.9271355353092, 1956.949688716373, 4.466173715600929, 0.9076766227592874, 275.53451501104354],
    ["RNN + RMSprop", 168, 24, 1431.3565403170057, 1999.2467893785217, 4.543059594197409, 0.9036425770705, 193.86489551253106],
    ["RNN + SGD", 168, 24, 1587.0868063796715, 2166.9122430922744, 5.075387416706246, 0.8868029761759146, 227.20396023915205],
    ["RNN + RMSprop + Learning Rate", 168, 24, 1391.4736401993944, 1955.1376315342836, 4.431959790886786, 0.9078475191143814, 260.42541081986445],
    ["RNN + Additional Layers", 168, 24, 1463.0599539297493, 2080.42019144102, 4.623851630921524, 0.8956591231413713, 287.09569205596506],
    ["RNN 128 Neurons", 168, 24, 1421.2446590304571, 2045.7797521954965, 4.48897927494526, 0.8991048910414456, 192.43200104890883],
    ["RNN Batch Size 32", 168, 24, 1407.5672331902024, 1977.5205726395573, 4.477985259160447, 0.9057254686072457, 311.5092754141123],
    ["RNN Batch Size 128", 168, 24, 1392.5691757587274, 1949.3553913471533, 4.4230654213894045, 0.9083917875401955, 237.11039008364617],
    ["RNN Hyperparameter Tuned", 168, 24, 1420.3538599296503, 2029.101753915076, 4.500508471335718, 0.9007432583397899, 219.11408073055358],
    ["RNN + Additional Lags", 168, 24, 1405.2472066353298, 1974.0998561917397, 4.474581326853382, 0.9061420038504027, 338.74023258018246]
]

rnn_168_24_df = pd.DataFrame(
    rnn_168_24_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

display(
    rnn_168_24_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

# Save inside existing comparison folder
os.makedirs("comparison", exist_ok=True)

rnn_168_24_df.to_csv(
    "comparison/RNN_168h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved: comparison/RNN_168h_to_24h_results.csv")

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,RNN Baseline,168,24,1430.213322,1981.727949,4.582685,0.905324,375.384036
1,RNN + EarlyStopping,168,24,1442.437417,1988.384035,4.570802,0.904687,85.233155
2,RNN + Dropout,168,24,1456.016175,2007.426075,4.653540,0.902853,206.212271
3,RNN + Batch Normalization,168,24,1449.425960,2022.758174,4.600892,0.901363,217.978013
4,RNN + Adam,168,24,1401.927136,1956.949689,4.466174,0.907677,275.534515
5,RNN + RMSprop,168,24,1431.356540,1999.246789,4.543060,0.903643,193.864896
6,RNN + SGD,168,24,1587.086806,2166.912243,5.075387,0.886803,227.203960
7,RNN + RMSprop + Learning Rate,168,24,1391.473640,1955.137632,4.431960,0.907848,260.425411
8,RNN + Additional Layers,168,24,1463.059954,2080.420191,4.623852,0.895659,287.095692
9,RNN 128 Neurons,168,24,1421.244659,2045.779752,4.488979,0.899105,192.432001


Saved: comparison/RNN_168h_to_24h_results.csv


In [2]:
best_model = rnn_168_24_df.loc[rnn_168_24_df["Model"] == "RNN + RMSprop + Learning Rate"]
display(best_model)

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
7,RNN + RMSprop + Learning Rate,168,24,1391.47364,1955.137632,4.43196,0.907848,260.425411
